This notebook is just for testing our data collection process for MI2. We’ll use it to pull a small sample of songs and genre information from MusicBrainz, add lyrics, and make sure our dataset setup actually works before we build the full version. It will also give us enough data to make the exploratory plots and describe the dataset for MI2.

In [1]:
import requests
import pandas as pd
import time
import os

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Set up Project 1 Folder

In [3]:
project_folder = "/content/drive/MyDrive/DS4002/Project 1"

os.makedirs(project_folder, exist_ok=True)

print("Project folder:", project_folder)

Project folder: /content/drive/MyDrive/DS4002/Project 1


Set up MusicBrainz

In [4]:
headers = {
    "User-Agent": "DS4002GenreProject/1.0 (mmr2ve@virginia.edu)"
}

base_url = "https://musicbrainz.org/ws/2/recording"

Creating API reuqest function

In [5]:
def get_musicbrainz_data(params, max_attempts=5):
    for attempt in range(max_attempts):

        response = requests.get(
            base_url,
            params=params,
            headers=headers,
            timeout=30
        )

        if response.status_code == 200:
            data = response.json()

            if "recordings" in data:
                print("Success!")
                print("Number of recordings returned:",
                      len(data["recordings"]))
                return data

        elif response.status_code == 503:
            print(
                f"MusicBrainz is busy. Waiting 5 seconds... "
                f"(attempt {attempt + 1}/{max_attempts})"
            )
            time.sleep(5)

        else:
            print("Unexpected error:", response.status_code)
            print(response.text)
            break

    raise Exception(
        "MusicBrainz request failed after multiple attempts."
    )

Request 100 country songs (test)

In [6]:
params = {
    "query": 'tag:"country"',
    "fmt": "json",
    "limit": 1000
}

data = get_musicbrainz_data(params)

Success!
Number of recordings returned: 25


Create dataframe

In [7]:
records = []

for r in data["recordings"]:

    artist_names = []
    artist_ids = []

    for credit in r.get("artist-credit", []):

        if isinstance(credit, dict):

            artist_names.append(
                credit.get("name", "")
            )

            artist_info = credit.get(
                "artist", {}
            )

            artist_ids.append(
                artist_info.get("id", "")
            )

    tags = [
        tag["name"]
        for tag in r.get("tags", [])
    ]

    records.append({
        "recording_id": r.get("id"),
        "song_title": r.get("title"),
        "artist": ", ".join(artist_names),
        "artist_id": ", ".join(artist_ids),
        "musicbrainz_tags": ", ".join(tags),
        "target_genre": "Country"
    })

country_df = pd.DataFrame(records)

country_df.head(10)

,recording_id,song_title,artist,artist_id,musicbrainz_tags,target_genre
0,10eddb05-abc1-4f23-8adf-321d7c91f245,Texico New Mexican Joe,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
1,eb0c9580-becd-4fea-b6bf-b5ea5cb53a72,Female Shuffle,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
2,a56949e3-01e0-4a1d-ac38-5c4c8a7d25dc,The Battery to My Heart,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
3,02a94584-757f-4d6a-ba6e-637ad5472c56,Mardi Grass in New Orleans,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
4,de364d46-fffb-4837-b3d2-0d5b427c3de5,Alarm Clock Boogie,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
5,5300d4dc-272f-4f07-9a8c-46188043bbdf,Amarilo Rose,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
6,0b77fb01-7da4-4fd9-8d4e-35da73ccd6a3,I'm Not Gonna Lock the Door,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country
7,f7c31ac9-4965-4cbe-a24d-703308cb2f2c,Old Friend,Lyle Lovett,7241e3ed-5ad4-4849-94df-6858ea833472,"country, country; alternative country",Country
8,297b4dcf-1291-43f7-ac9f-4821d76f379b,Love Will,Trace Adkins,3dc9c919-508f-4255-b05e-0eed6a28137d,"country, country; modern country",Country
9,7dcdb956-e651-4f2c-ad1f-64f8584db748,A Place in the Sunshine,Billy Briggs & His X.I.T. Boys,dafd91a6-1b50-4ffa-af0e-0168360d5ee6,"country, country/country swing",Country


In [8]:
print("Number of recordings:", len(country_df))
print("Number of unique artist credits:",
      country_df["artist"].nunique())

Number of recordings: 25
Number of unique artist credits: 4


In [9]:
country_df["artist"].value_counts().head(20)

,count
artist,
Billy Briggs & His X.I.T. Boys,22
Lyle Lovett,1
Trace Adkins,1
C.W. McCall,1


Find artists with at least 3 songs

In [10]:
artist_counts = country_df["artist"].value_counts()

eligible_country_artists = (
    artist_counts[artist_counts >= 3]
)

print(
    "Number of eligible Country artists:",
    len(eligible_country_artists)
)

eligible_country_artists

Number of eligible Country artists: 1


,count
artist,
Billy Briggs & His X.I.T. Boys,22


3 random artists

In [11]:
pilot_country_artists = (
    eligible_country_artists
    .sample(
        n=5,
        random_state=4002
    )
    .index
)

pilot_country_artists

ValueError: Cannot take a larger sample than population when 'replace=False'

3 random songs from each artist

In [ ]:
country_pilot = (
    country_df[
        country_df["artist"].isin(
            pilot_country_artists
        )
    ]
    .groupby(
        "artist",
        group_keys=False
    )
    .sample(
        n=3,
        random_state=4002
    )
)

country_pilot

In [ ]:
country_pilot["artist"].value_counts()

In [ ]:
print("Total pilot songs:", len(country_pilot))
print("Artists represented:",
      country_pilot["artist"].nunique())

Saving Pilot to Drive

In [ ]:
country_file = project_folder + "/country_pilot.csv"

country_pilot.to_csv(
    country_file,
    index=False
)

print("Saved to:", country_file)

In [ ]:
project_folder = "/content/drive/MyDrive/UVA 2026 27/DS4002/Project 1"

In [ ]:
country_file = project_folder + "/country_pilot.csv"

country_pilot.to_csv(country_file, index=False)

print("Saved to:", country_file)

In [ ]:
def get_listenbrainz_popularity(artist_mbid):
    payload = {
        "artist_mbids": [artist_mbid]
    }

    response = requests.post(
        listenbrainz_url,
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        return None, None

    result = response.json()

    if not result:
        return None, None

    return (
        result[0].get("total_listen_count"),
        result[0].get("total_user_count")
    )

In [ ]:
listenbrainz_url = "https://api.listenbrainz.org/1/popularity/artist"

Country Pilot works, now doing the same for Pop, Rock, and then Hip-Hop/Rap

In [ ]:
def make_genre_pilot(search_tag, target_genre, min_users=10000):

    # --------------------------------
    # 1. Get 500 candidate recordings
    # --------------------------------
    all_recordings = []

    for offset in range(0, 500, 100):

        params = {
            "query": f'tag:"{search_tag}"',
            "fmt": "json",
            "limit": 100,
            "offset": offset
        }

        data = get_musicbrainz_data(params)

        all_recordings.extend(
            data["recordings"]
        )

        # Be polite to MusicBrainz
        time.sleep(1.1)

    print(
        "Total raw recordings retrieved:",
        len(all_recordings)
    )

    # --------------------------------
    # 2. Convert results to dataframe
    # --------------------------------
    records = []

    for r in all_recordings:

        artist_names = []
        artist_ids = []

        for credit in r.get("artist-credit", []):

            if isinstance(credit, dict):

                artist_names.append(
                    credit.get("name", "")
                )

                artist_info = credit.get(
                    "artist", {}
                )

                artist_ids.append(
                    artist_info.get("id", "")
                )

        tags = [
            tag["name"]
            for tag in r.get("tags", [])
        ]

        records.append({
            "recording_id": r.get("id"),
            "song_title": r.get("title"),
            "artist": ", ".join(artist_names),
            "artist_id": ", ".join(artist_ids),
            "musicbrainz_tags": ", ".join(tags),
            "target_genre": target_genre
        })

    df = pd.DataFrame(records)

    # Remove duplicate recordings
    df = df.drop_duplicates(
        subset="recording_id"
    ).reset_index(drop=True)

    print("Genre:", target_genre)
    print("Candidate songs:", len(df))
    print(
        "Unique artist credits:",
        df["artist"].nunique()
    )

    # --------------------------------
    # 3. Require at least 3 songs
    # --------------------------------
    artist_counts = (
        df["artist"].value_counts()
    )

    eligible_artists = artist_counts[
        artist_counts >= 3
    ]

    print(
        "Artists with at least 3 songs:",
        len(eligible_artists)
    )

    # --------------------------------
    # 4. Make artist table
    # --------------------------------
    artist_df = (
        df[
            df["artist"].isin(
                eligible_artists.index
            )
        ][["artist", "artist_id"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Exclude collaborations/multiple artist IDs
    artist_df = artist_df[
        ~artist_df["artist_id"].str.contains(
            ",",
            na=False
        )
    ].copy()

    # --------------------------------
    # 5. Get ListenBrainz popularity
    # --------------------------------
    listen_counts = []
    user_counts = []

    for artist_id in artist_df["artist_id"]:

        listens, users = (
            get_listenbrainz_popularity(
                artist_id
            )
        )

        listen_counts.append(listens)
        user_counts.append(users)

        time.sleep(0.5)

    artist_df["listen_count"] = (
        listen_counts
    )

    artist_df["user_count"] = (
        user_counts
    )

    # --------------------------------
    # 6. Apply popularity cutoff
    # --------------------------------
    popular_artists = artist_df[
        artist_df["user_count"] >= min_users
    ].copy()

    print(
        f"Artists with >= {min_users:,} "
        f"ListenBrainz users:",
        len(popular_artists)
    )

    display(
        popular_artists.sort_values(
            "user_count",
            ascending=False
        )
    )

    # --------------------------------
    # 7. Need at least 5 artists
    # --------------------------------
    if len(popular_artists) < 5:

        print(
            "Still not enough qualifying "
            "artists for this genre."
        )

        return None

    # --------------------------------
    # 8. Randomly select 5 artists
    # --------------------------------
    selected_artists = (
        popular_artists
        .sample(
            n=5,
            random_state=4002
        )["artist"]
    )

    print("\nSelected artists:")
    print(selected_artists.tolist())

    # --------------------------------
    # 9. Select 3 songs per artist
    # --------------------------------
    pilot = (
        df[
            df["artist"].isin(
                selected_artists
            )
        ]
        .groupby(
            "artist",
            group_keys=False
        )
        .sample(
            n=3,
            random_state=4002
        )
        .reset_index(drop=True)
    )

    return pilot

Creating Pop


In [ ]:
pop_pilot = make_genre_pilot(
    "pop",
    "Pop"
)

In [ ]:
pop_pilot["artist"].value_counts()

In [ ]:
pop_pilot

Note that YUI is a japanese artist, so down the line, will need to get an English-speaking one, as translating adds another layer of complexity.

Saving Pop

In [ ]:
pop_file = project_folder + "/pop_pilot.csv"

pop_pilot.to_csv(
    pop_file,
    index=False
)

print("Saved to:", pop_file)

In [ ]:
time.sleep(2)

Creating Rock

In [ ]:
rock_pilot = make_genre_pilot(
    "rock",
    "Rock"
)

In [ ]:
rock_pilot["artist"].value_counts()

In [ ]:
rock_pilot

In [ ]:
rock_file = project_folder + "/rock_pilot.csv"

rock_pilot.to_csv(
    rock_file,
    index=False
)

print("Saved to:", rock_file)

Finally Hip-Hop/Rap

In [ ]:
hiphop_pilot = make_genre_pilot(
    "hip hop",
    "Hip-Hop/Rap"
)

In [ ]:
hiphop_pilot["artist"].value_counts()

In [ ]:
hiphop_pilot[
    ["artist", "song_title", "musicbrainz_tags", "target_genre"]
].sort_values("artist")

Save ALL songs from all genres to one, 60 song CSV

In [ ]:
country_file = project_folder + "/country_pilot.csv"
pop_file = project_folder + "/pop_pilot.csv"
rock_file = project_folder + "/rock_pilot.csv"
hiphop_file = project_folder + "/hiphop_pilot.csv"

country_pilot.to_csv(country_file, index=False)
pop_pilot.to_csv(pop_file, index=False)
rock_pilot.to_csv(rock_file, index=False)
hiphop_pilot.to_csv(hiphop_file, index=False)

print("Saved:")
print(country_file)
print(pop_file)
print(rock_file)
print(hiphop_file)

In [ ]:
pilot_df = pd.concat(
    [
        country_pilot,
        pop_pilot,
        rock_pilot,
        hiphop_pilot
    ],
    ignore_index=True
)

pilot_df

In [ ]:
print("Total songs:", len(pilot_df))
print("Genres:", pilot_df["target_genre"].value_counts())

In [ ]:
combined_file = project_folder + "/music_genre_pilot.csv"

pilot_df.to_csv(
    combined_file,
    index=False
)

print("Saved combined pilot to:", combined_file)

In [ ]:
check = pd.read_csv(combined_file)

print(check.shape)
check["target_genre"].value_counts()

In [ ]:
combined_file = project_folder + "/music_genre_pilot.csv"

check = pd.read_csv(combined_file)

print("Total songs:", len(check))
print()
print(check["target_genre"].value_counts())

In [ ]:
check.groupby("target_genre")["artist"].nunique()